# Bird Training Pipeline (CUB-200-2011)
EfficientNet-B0 transfer learning notebook with artifact export to backend models directory.

In [ ]:
from __future__ import annotations

import json
import os
import random
import shutil
from pathlib import Path

import gdown
import torch
import torch.nn as nn
from PIL import Image
from sklearn.model_selection import train_test_split
from torch.utils.data import DataLoader, Dataset
from torchvision import models, transforms

SEED = 42
random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

XLA_XM = None

def detect_best_device() -> tuple[torch.device, str, bool]:
    """Auto-select the best available accelerator: TPU > CUDA > MPS > CPU."""
    global XLA_XM

    # TPU (PyTorch/XLA)
    try:
        import torch_xla.core.xla_model as xm  # type: ignore
        XLA_XM = xm
        return xm.xla_device(), 'tpu', True
    except Exception:
        XLA_XM = None

    # NVIDIA CUDA GPU
    if torch.cuda.is_available():
        return torch.device('cuda'), 'cuda', False

    # Apple Silicon GPU (MPS)
    if hasattr(torch.backends, 'mps') and torch.backends.mps.is_available():
        return torch.device('mps'), 'mps', False

    # CPU fallback
    return torch.device('cpu'), 'cpu', False

DEVICE, DEVICE_TYPE, IS_TPU = detect_best_device()
USE_CUDA = DEVICE_TYPE == 'cuda'
USE_MPS = DEVICE_TYPE == 'mps'

if USE_CUDA:
    torch.backends.cudnn.benchmark = True  # faster conv kernels for fixed input shapes
    torch.set_float32_matmul_precision('high')
elif USE_MPS:
    torch.set_float32_matmul_precision('high')

print(f'Using device: {DEVICE} ({DEVICE_TYPE.upper()})')

In [ ]:
DATA_DIR = Path('data')
DATASET_DIR = DATA_DIR / 'CUB_200_2011'
ARCHIVE_PATH = DATA_DIR / 'CUB_200_2011.tgz'

DATA_DIR.mkdir(parents=True, exist_ok=True)

# Public mirrors can change; update URL if needed.
CUB_URL = 'https://data.caltech.edu/records/65de6-vp158/files/CUB_200_2011.tgz?download=1'

if not DATASET_DIR.exists():
    if not ARCHIVE_PATH.exists():
        print('Downloading CUB-200-2011...')
        gdown.download(CUB_URL, str(ARCHIVE_PATH), quiet=False, fuzzy=True)

    print('Extracting dataset...')
    import tarfile
    with tarfile.open(ARCHIVE_PATH, 'r:gz') as tar:
        tar.extractall(DATA_DIR)

print('Dataset path:', DATASET_DIR.resolve())

In [ ]:
images_txt = DATASET_DIR / 'images.txt'
labels_txt = DATASET_DIR / 'image_class_labels.txt'
classes_txt = DATASET_DIR / 'classes.txt'
images_root = DATASET_DIR / 'images'

image_id_to_path = {}
with images_txt.open('r', encoding='utf-8') as f:
    for line in f:
        idx, rel_path = line.strip().split(' ', 1)
        image_id_to_path[int(idx)] = rel_path

image_id_to_label = {}
with labels_txt.open('r', encoding='utf-8') as f:
    for line in f:
        idx, cls = line.strip().split(' ', 1)
        image_id_to_label[int(idx)] = int(cls) - 1  # zero-based

class_names = []
with classes_txt.open('r', encoding='utf-8') as f:
    for line in f:
        _, class_name = line.strip().split(' ', 1)
        class_names.append(class_name.replace('_', ' '))

samples = []
for idx, rel_path in image_id_to_path.items():
    label = image_id_to_label[idx]
    samples.append((images_root / rel_path, label))

print('Samples:', len(samples), 'Classes:', len(class_names))

In [ ]:
train_samples, val_samples = train_test_split(
    samples,
    test_size=0.2,
    random_state=SEED,
    stratify=[label for _, label in samples],
)

train_tfms = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

val_tfms = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

class BirdDataset(Dataset):
    def __init__(self, rows, transform):
        self.rows = rows
        self.transform = transform

    def __len__(self):
        return len(self.rows)

    def __getitem__(self, idx):
        path, label = self.rows[idx]
        image = Image.open(path).convert('RGB')
        return self.transform(image), label

train_ds = BirdDataset(train_samples, train_tfms)
val_ds = BirdDataset(val_samples, val_tfms)

cpu_count = os.cpu_count() or 2
if IS_TPU:
    num_workers = 0  # safer default in notebook TPU flows
elif DEVICE_TYPE in {'cuda', 'mps'}:
    num_workers = min(8, max(2, cpu_count // 2))
else:
    num_workers = min(4, max(1, cpu_count - 1))

pin_memory = USE_CUDA
persistent_workers = num_workers > 0 and not IS_TPU

train_loader = DataLoader(
    train_ds,
    batch_size=32,
    shuffle=True,
    num_workers=num_workers,
    pin_memory=pin_memory,
    persistent_workers=persistent_workers,
    drop_last=True,
    prefetch_factor=2 if num_workers > 0 else None,
)
val_loader = DataLoader(
    val_ds,
    batch_size=64,
    shuffle=False,
    num_workers=num_workers,
    pin_memory=pin_memory,
    persistent_workers=persistent_workers,
    prefetch_factor=2 if num_workers > 0 else None,
)

print(
    f'Train batches: {len(train_loader)} | Val batches: {len(val_loader)} | '
    f'workers={num_workers} | pin_memory={pin_memory}'
)

In [ ]:
model = models.efficientnet_b0(weights=models.EfficientNet_B0_Weights.IMAGENET1K_V1)
in_features = model.classifier[1].in_features
model.classifier[1] = nn.Linear(in_features, 200)
model = model.to(DEVICE)

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4)

scaler = torch.amp.GradScaler('cuda', enabled=USE_CUDA)

def evaluate(loader):
    model.eval()
    total_loss, correct, total = 0.0, 0, 0
    # NOTE: no_grad is safer than inference_mode here;
    # inference_mode can trigger "Cannot set version_counter for inference tensor"
    # with some model/operator/runtime combinations.
    with torch.no_grad():
        for x, y in loader:
            x = x.to(DEVICE, non_blocking=USE_CUDA)
            y = y.to(DEVICE, non_blocking=USE_CUDA)

            with torch.amp.autocast(device_type='cuda', enabled=USE_CUDA):
                logits = model(x)
                loss = criterion(logits, y)

            total_loss += loss.item() * x.size(0)
            preds = logits.argmax(dim=1)
            correct += (preds == y).sum().item()
            total += y.size(0)
    return total_loss / total, correct / total

best_acc = 0.0 # Initialize best accuracy
epochs = 8  # Increase for stronger accuracy

for epoch in range(1, epochs + 1):
    model.train()
    for x, y in train_loader:
        x = x.to(DEVICE, non_blocking=USE_CUDA)
        y = y.to(DEVICE, non_blocking=USE_CUDA)

        optimizer.zero_grad(set_to_none=True)

        with torch.amp.autocast(device_type='cuda', enabled=USE_CUDA):
            logits = model(x)
            loss = criterion(logits, y)

        if USE_CUDA:
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
        else:
            loss.backward()
            if IS_TPU and XLA_XM is not None:
                XLA_XM.optimizer_step(optimizer)
            else:
                optimizer.step()

    val_loss, val_acc = evaluate(val_loader)
    print(f'Epoch {epoch:02d} | val_loss={val_loss:.4f} | val_acc={val_acc:.4f}')

    if val_acc > best_acc:
        best_acc = val_acc
        torch.save(model.state_dict(), 'bird_model.pth')

print('Best validation accuracy:', best_acc)
target_acc = 0.70
metrics = {'best_val_accuracy': best_acc, 'target_accuracy': target_acc, 'device_type': DEVICE_TYPE}
Path('training_metrics.json').write_text(json.dumps(metrics, indent=2), encoding='utf-8')
if best_acc < target_acc:
    print(f'⚠️ Validation accuracy {best_acc:.4f} is below target {target_acc:.2f}. Consider more epochs/tuning.')

In [ ]:
Path('label_list.json').write_text(json.dumps(class_names, indent=2), encoding='utf-8')
print('Saved bird_model.pth and label_list.json in training directory')

In [ ]:
# Define and create the path explicitly
local_backend_path = Path('../apps/backend/models')
local_backend_path.mkdir(parents=True, exist_ok=True)

# Check if files were successfully saved
model_exists = Path('bird_model.pth').exists()
labels_exists = Path('label_list.json').exists()

print(f"Current Working Directory: {os.getcwd()}")
print(f"Target Backend Path: {local_backend_path.resolve()}")
print(f"Source bird_model.pth exists: {model_exists}")
print(f"Source label_list.json exists: {labels_exists}")

if model_exists and labels_exists:
    print("\nReady to sync. If VS Code still fails, try refreshing the file explorer or manually downloading the files from the Colab sidebar.")